# PySpark y Spark SQL

PySpark es la interfaz de Python de Apache Spark. En esta práctica se lee la serie de COVID-19 y se consulta con DataFrames y con SQL.

Un filtro, una agrupación o un orden se escriben en una frase SQL. La referencia de la API está en http://spark.apache.org/docs/latest/api/python/


# 1. Sesión de Spark


In [ ]:
%pip install pyspark

from pyspark.sql import SparkSession
import pyspark.sql.functions as F


spark = (
    SparkSession.builder
    .appName("6.2 Spark y SQL")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

sc = spark.sparkContext
spark

# 2. Lectura de datos

La serie es la de Our World in Data sobre COVID-19. El archivo `owid-covid-data.csv` ya está en la carpeta de la notebook. Cada fila es un país, o un agregado, en una fecha.


In [ ]:
# import requests
# path = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
# req = requests.get(path)
# url_content = req.content
#
# csv_file_name = 'owid-covid-data.csv'
# csv_file = open(csv_file_name, 'wb')
#
# csv_file.write(url_content)
# csv_file.close()

df = spark.read.csv("owid-covid-data.csv", header=True, inferSchema=True)

# 3. DataFrames


In [ ]:
# Nombre y tipo de cada columna
df.printSchema()

In [ ]:
# date llega como texto. Esta consulta la muestra como fecha y no modifica df.
df.select(F.to_date(df.date).alias('date'))

In [ ]:
# Resumen de las columnas numéricas
df.describe().show()

In [ ]:
# Argentina, de la fecha más reciente a la más antigua
df.filter(df.location == "Argentina").orderBy(F.desc("date")).show()

In [ ]:
# Suma de new_cases por location
df.groupBy("location").sum("new_cases").orderBy(F.desc("sum(new_cases)")).show(truncate=False)

# 4. Spark SQL

`createOrReplaceTempView` publica el DataFrame como una vista de esta sesión. A partir de ahí la consulta se escribe en SQL.


In [ ]:
# Vista temporal de esta sesión
df.createOrReplaceTempView("covid_data")
# Para guardarla en el catálogo:
# df.write.mode("overwrite").saveAsTable("covid_data")


In [ ]:
df2 = spark.sql("SELECT * from covid_data")
df2.printSchema()

In [ ]:
df2.show()

In [ ]:
# Cantidad de filas por location
groupDF = spark.sql("SELECT location, count(*) from covid_data group by location")
groupDF.show()


### Ejercicio A. Total de casos confirmados por país hasta la fecha más reciente


In [ ]:
query = """
SELECT location AS pais, SUM(total_cases) AS total_casos_confirmados
FROM covid_data
GROUP BY location
ORDER BY total_casos_confirmados DESC
LIMIT 10
"""

result = spark.sql(query)
result.show()

### Ejercicio B. Número de muertes el 2022-01-01


In [ ]:
query = """
TODO
"""

result = spark.sql(query)
result.show()

### Ejercicio C. El día con más muertes nuevas en Argentina

Devolver la fecha y `new_deaths` de ese día.


In [ ]:
query = """
TODO
"""

result = spark.sql(query)
result.show()


### Ejercicio D. Países por continente

Contar cuántos `location` distintos hay en cada `continent`. Dejar afuera las filas en las que `continent` es nulo.


In [ ]:
query = """
TODO
"""

result = spark.sql(query)
result.show()
